# EDA do AlertaRio

Analise visual dos artefatos produzidos pela auditoria versionada do AlertaRio. Este notebook nao recalcula regras de qualidade a partir dos Parquets brutos.

## Pre-requisito

Execute antes `nowcasting-audit-alertario --input-root data/pluviometricos_alertario --output-dir outputs/analysis/alertario/raw_audit_v1`. Os CSVs e o JSON resultantes sao a fonte unica das metricas de qualidade exibidas abaixo.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Nao foi possivel localizar a raiz do repositorio.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DEFAULT_AUDIT_DIR = PROJECT_ROOT / 'outputs' / 'analysis' / 'alertario' / 'raw_audit_v1'
AUDIT_DIR = Path(os.environ.get('ALERTARIO_AUDIT_DIR', DEFAULT_AUDIT_DIR))
MAPPING_PATH = PROJECT_ROOT / 'data' / 'mapeamento_pixel_estacao_alertario.csv'

required_files = ('summary.json', 'station_summary.csv', 'station_year_summary.csv')
missing = [name for name in required_files if not (AUDIT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f'Artefatos de auditoria ausentes em {AUDIT_DIR}: {missing}. ' 
        'Execute nowcasting-audit-alertario antes de abrir este notebook.'
    )

print(f'Projeto: {PROJECT_ROOT}')
print(f'Auditoria: {AUDIT_DIR}')

In [ ]:
with (AUDIT_DIR / 'summary.json').open(encoding='utf-8') as handle:
    audit_summary = json.load(handle)

station_summary = pd.read_csv(AUDIT_DIR / 'station_summary.csv')
station_year_summary = pd.read_csv(AUDIT_DIR / 'station_year_summary.csv')

if MAPPING_PATH.is_file():
    mapping = pd.read_csv(MAPPING_PATH, usecols=['station_id', 'nome'])
    station_summary = station_summary.merge(mapping, on='station_id', how='left')
else:
    station_summary['nome'] = pd.NA

station_summary['estacao'] = station_summary['nome'].fillna(
    'ID ' + station_summary['station_id'].astype(str)
)
station_summary['invalid_or_missing'] = (
    station_summary['missing_m15']
    + station_summary['sentinel_m15']
    + station_summary['negative_m15']
)
station_summary['invalid_or_missing_pct'] = 100 * station_summary['invalid_or_missing'] / station_summary['rows']
station_summary['suspect_pct'] = 100 * station_summary['suspect_m15'] / station_summary['valid_m15']
station_summary['reject_pct'] = 100 * station_summary['reject_m15'] / station_summary['valid_m15']

print('Arquivos auditados:', audit_summary['files'])
print('Estacoes:', audit_summary['stations'])
print('Periodo:', min(audit_summary['years']), 'a', max(audit_summary['years']))

## Resumo global de qualidade

In [ ]:
global_metrics = pd.Series(audit_summary['totals'], name='registros').to_frame()
global_metrics['percentual_dos_registros'] = 100 * global_metrics['registros'] / global_metrics.loc['rows', 'registros']
display(global_metrics)

## Cobertura temporal

In [ ]:
coverage_by_year = (
    station_year_summary[station_year_summary['valid_m15'] > 0]
    .groupby('year')['station_id']
    .nunique()
    .rename('estacoes_com_observacao_valida')
    .reset_index()
)

ax = coverage_by_year.plot(x='year', y='estacoes_com_observacao_valida', kind='bar', legend=False, figsize=(12, 4))
ax.set(xlabel='Ano', ylabel='Estacoes com m15 valido', title='Cobertura anual do AlertaRio')
plt.tight_layout()
display(coverage_by_year)

In [ ]:
required_years = set(range(2012, 2025))
years_by_station = (
    station_year_summary[station_year_summary['valid_m15'] > 0]
    .groupby('station_id')['year']
    .agg(lambda years: sorted(set(years)))
    .rename('anos_com_observacao_valida')
    .reset_index()
)
coverage_table = station_summary[['station_id', 'estacao', 'first_timestamp', 'last_timestamp']].merge(
    years_by_station, on='station_id', how='left'
)
coverage_table['cobre_2012_2024'] = coverage_table['anos_com_observacao_valida'].map(
    lambda years: required_years.issubset(set(years)) if isinstance(years, list) else False
)
print('Estacoes com cobertura em todos os anos de 2012-2024:', int(coverage_table['cobre_2012_2024'].sum()))
display(coverage_table.sort_values(['cobre_2012_2024', 'station_id'], ascending=[False, True]))

## Estacoes que exigem inspecao

In [ ]:
quality_columns = [
    'station_id', 'estacao', 'rows', 'valid_m15', 'invalid_or_missing',
    'invalid_or_missing_pct', 'suspect_m15', 'suspect_pct',
    'reject_m15', 'reject_pct', 'max_valid_m15',
]
quality_table = station_summary[quality_columns].sort_values(
    ['reject_m15', 'suspect_m15', 'invalid_or_missing'], ascending=False
)
display(quality_table)

ax = quality_table.head(15).plot.barh(
    x='estacao', y=['suspect_m15', 'reject_m15'], stacked=True, figsize=(10, 6)
)
ax.set(xlabel='Registros', ylabel='Estacao', title='15 estacoes com mais registros suspeitos ou rejeitados')
plt.tight_layout()

## Limites e proximos passos

- Consulte `docs/CONTROLE_QUALIDADE_ALERTARIO.md` para as regras e limites usados na auditoria.
- O nome e preenchido apenas para as estacoes presentes no mapeamento espacial atual; os demais IDs sao exibidos como `ID <n>`.
- Use `notebooks/02_geospatial/01_mapa_estacoes_alertario.ipynb` para a distribuicao espacial.
- A construcao de targets e qualquer mudanca de regra de qualidade permanecem nos comandos versionados do pipeline.